# **Scientific Python: NumPy**

These notebooks are my personal notes on **NumPy,** the base library for numerical computing in Python. Almost the whole scientific Python stack (pandas, SciPy, Matplotlib, scikit-learn…) is built on top of it.

The plan is to go topic by topic, one notebook each, combining explanations with code examples I can come back to later. All the documentation used will be found at the end of each notebook. 

This first notebook covers the core object of the library: the **`ndarray`** (N-dimensional array). That means what it is, how to create one, and why it's faster and more convenient than plain Python lists for numerical work.

## **NumPy**

NumPy (**Num**erical **Py**thon) is the essential library for numerical, scientific computing in Python. It contains multidimensional array data structues, most importantly the **``ndarray``** and a collection of operations that operate efficiently on them.

NumPy provides an efficient interface for storage and manipulation of datasets, which can be represented many times as **arrays** of **numbers.** 

### **Installation**

To install (the most recent version of) NumPy into a virtual environment, one can simply use `pip`:

```python
pip install numpy
```

To verify the installation:

In [ ]:
import numpy as np # "np" is used as a widespread convention to access NumPy's features. 

print(np.__version__)

## **NumPy Arrays: The `ndarray`**

The most important concept when using NumPy is the `ndarray` **object.** This type of object is made of *n*-dimensional **(multidimensional)** arrays of homogeneous data types. 

In CS, an array is a structure to store and retrieve data, usually represented as a grid with cells, each one storing one data element. A one-dimensional array would be like a list; a two-dimensional one, a table; and a three-dimensional one, a collection of tables. The `ndarray` gets this idea and extends it to any number ***n*** of dimensions.

But, compared to the common Python sequence data types, they have some catches:

- When created, the **size** is **fixed.** Once created, the total size cannot change. Changing its size will create a new array, deleting the original.
- `ndarray`s require its elements to be of the **same** datatype (`dtype`): same **size** in memory.
- The shape must be **rectangular:** each row must have the same number of columns.

With these, they make operations on large datasets faster, more memory efficient, and requiring less code than Python's built-in sequences or other less restrictive structures.


### **Why does `ndarray` exist?**

**Limitations of Python lists for numerical work**

Python lists are very flexible: they can grow and shrink dynamically, and hold any types of objects. That flexibility costs a lot when the job is crunching numbers:

- Lists cannot do element-wise operations. To perform operations on every element you need an explicit loop or a comprehension.
- That loop runs in the interpreter, one element at a time. On every step Python has to check what type each element is and find the right operation for it, because nothing guarantees the next element is the same type as the last one.

As a simple example, consider the case of multiplying each element in a 1-D sequence with the corresponding element in another sequence of the same length. If the data are stored in two Python lists, `a` and `b`, we could iterate over each element:

```python
c = []
for i in range(len(a)):
    c.append(a[i]*b[i])
```
Which is a valid option, but when lists grow in size, using loops to operate element-wise will become inefficient. The code work will as well increase with the dimensionality of our data.

**Homogeneous arrays vs lists of objects**

A Python list doesn't hold the numbers themselves. It holds **pointers,** (-->) each one pointing to a separate Python object somewhere in memory. Each of those objects (even a small `int`) carries its own header (a reference count and a pointer to its type) on top of the actual value.

Again, the good thing about them is flexibility, but it comes at a cost: each item must contain its type, reference count and other info. But when every item is of the same type, the info. becomes redundant and a fixed-type array would be more efficient.

An `ndarray` is **homogeneous:** every element has the same `dtype`. The type is stored **once** for the whole array, and the elements are only the raw values, with no per-element object overhead. For its implementation, the array only contains single pointer to one **contiguous** block of data; a list contains a pointer to a block of pointers, each pointing to an object.

So, NumPy arrays lack the flexibility of list, but are more efficient in data storage and manipulation.

**Contiguous memory storage**

According to the official docs: *`ndarray`: "a contiguous one-dimensional segment of computer memory (owned by the array, or by some other object), combined with an indexing scheme that maps N integers into the location of an item in the block."*

In other words: it is one block of memory (the **data buffer**) plus metadata that says how to read it (`dtype`, `shape`...). Since all elements are the same size and sit next to each other, NumPy can compute where any element is directly, without following pointers. That allows:

- **Vectorization:** operations are computed in the absence of explicit looping, behind the scenes in pre-compiled C code. I will come back later to this in its own notebook.
- Better use of the CPU, which loads neighbouring memory in chunks.

### **Array Basics**

NumPy's class for array object creation is `ndarray`, or simply "array". There are many ways to initialize a NumPy array.

The most straightforward is to create one using any Python sequence, such as a list or tuple:

In [ ]:
a = np.array([5, 3])
a

In [ ]:
a = np.array((0, 1, 2))
a

`array` should not be called with multiple arguments: a single sequence should only be provided:

In [ ]:
try:
    np.array(0, 1, 2)
except Exception as e:
    print(repr(e))

2D arrays can be created from a sequence of sequences: 3D, from a sequence of sequences of sequences; and so on:

In [ ]:
b = np.array([[0, 1, 2], [3, 4, 5]])
b

NumPy arrays, as said before, can only contain items of the same **data type.** If they do not match, or the type is not given, NumPy will try to use one that can represent all the values by applying its **promotion** rules. For example:

In [ ]:
b = np.array([True, 3, 2.5])  # Boolean and integer upscaled to float (can represent all of them).
b

### **Array Attributes**

An array object has different **attributes** that provide core information of the array itself. They can be used to **get** or even **set** the properties to an array without creating a new one. Depending on the type of information they contain, there are:

#### **Memory-Layout Attributes**

Provide information about how an array's elements are stored in memory. The most useful are:

| Attribute | Description |
|---|---|
| `ndarray.ndim` | Number of array axes (dimensions). |
| `ndarray.shape` | The array's dimensions. Tuple of `int`s indicating the size of array in each dimension. For a 2D array with (m, n) rows and columns, `shape` will be `(m, n)`. |
| `ndarray.size` | Total number of elements in the array. = to product of its dimensions (`shape`). |
| `ndarray.itemsize` | Size of each array element in B. |

In [ ]:
x = np.array([[1, 2, 3], [4, 5, 6]])

print(x.ndim)  # 2 dimensions, 2 axes
print(x.shape) # 2 rows, 3 columns
print(x.size)  # 2r x 3c -> 6
print(x.itemsize) # 8 B. According to the dtype, it will change. More on that later.

#### **Data-Type**

The **`dtype`** attribute of an **`ndarray`** provides the data-type object of the elements of the array. Describes the type of data stored in an array. Arrays are homogeneous, so they contain elements of the **same** data-type, contained in this attribute:

In [ ]:
x.dtype

Every `ndarray` element takes up the same size slot in memory, and all slots are interpreted in the same way, dictated by this attribute.

#### **Other Attributes**

The `ndarray` object contains a many other attributes; some of them that can result useful are:

- **`ndarray.data`:** provides the buffer that contains the actual elements of the array in memory. Like a **pointer** to the memory location where they are stored:

In [ ]:
arr1 = np.array([1, 2])
arr2 = np.array([0, 3])

print("arr1 at: ",arr1.data)
print("arr2 at: ",arr2.data)

- **`ndarray.real`, `ndarray.imag`:** real and imaginary part of the array elements.
- **`ndarray.T`:** provides a **view** of the **transposed** array.

### **Array Creation Methods**

Apart from defining arrays from Python built-in sequence data types (lists, tuples), there are other ways to create arrays, which may be more suitable in other situations.

#### **Built-in NumPy Functions**

NumPy has a good amount of built-in functions (or routines) for creating arrays of any dimensionality. Depending on this number, they can be grouped as:

##### **1D array Creation Functions**

The most important ones are `np.arange` and `np.linspace`.

- **`np.arange`** creates an array with evenly spaced **incrementing** values. Needs at least an `stop` argument. Its complete list of parameters is:
    ```python
    numpy.arange([start, ]stop, [step, ]dtype=None, *, device=None, like=None)
    ```
    
<div style="margin-left: 40px;">

| Argument | Meaning |
|---|---|
| `start` | Start of the interval. Value included. (Optional: default = 0.)|
| `stop` | End of the interval. Value not included. |
| `step` | Increment between values. (Optional: default = 1.) |
| `dtype` | Dtype of output array. (Optional: if not given, infer it.) |

So `arange` produces either a `[0, stop)`, `[start, stop)` or `[start, (step) stop)` interval. For example: 
</div>

In [ ]:
x = np.arange(5)  # Only 'stop' parameter defined.
print(x)

x = np.arange(1, 10, 2)  # 'start' + 'step' defined. Both are optional. 
print(x)

x = np.arange(1, 5, 0.7)  # Floating point number as 'step'.
print(x)

x = np.arange(-1, 2, 0.2, dtype=np.int_)  # Unexpected behaviour. Floating point value as 'step', but
# the dtype of the output array is forced to integer. The step value is converted to an int:
# int(0.2) -> 0, but the default is 1, the value which we end with. Also, the interval did not end at
#'stop' = 2.
print(x)

However, using floating point number as arguments, especially for `step`, can result in `stop` being included in the output, due to floating point overflow/roundoff errors. The number of elements of the output array will be `ceil((stop-start)/step)`.

The best practice is to use all values as **integers.** When using non-integers, `linspace` should be used.

- **`np.linspace`** creates an array with *n* **evenly spaced** values over a specified **interval.** Needs at least an `start` and `stop` argument. The set of arguments the function accepts is:
  
    ```python
    numpy.linspace(start, stop, num=50, endpoint=True, retstep=False, dtype=None, axis=0, *, device=None)
    ```
<div style="margin-left: 40px;">
The most relevant ones are explained as follows:

<br>
<br>

| Argument | Meaning |
|---|---|
| `start` | Start of the interval. Value included.|
| `stop` | End of the interval. Value included, unless `endpoint=False` (step size changes). |
| `num` | Number of values to generate, non-negative. (Optional: default = 50.) |
| `endpoint` | If False, `stop` is excluded. (Optional: default = True.) |

With `linspace`, intervals of the type `[start, stop]` or `[start, stop)` are generated as arrays. 

As an example, let's try:
</div>

In [ ]:
x = np.linspace(0, 1, num=5)  # (Five) Evenly spaced values in the interval [0, 5].
print(x)

x = np.linspace(0, 1, num=5, endpoint=False)  # (Three) evenly spaced values in the same interval. 
# Endpoint (5) excluded.
print(x)

This function has better endpoint handling than `arange`. Also, if `dtype` is not given, it will be inferred, but never to an integer: **float** will be used even if the array produced was of integers.

##### **2D array Creation Functions**

They are used to create 2D special matrices, represented as 2D arrays. They are:

- **`np.eye(n, m)`** generates an 2D n x m **identity** matrix, with 1s on the diagonal, and 0s elsewhere:

In [ ]:
x = np.eye(4, 6)  # Identity matrix with n=4 rows and m=6 columns.
print(x)

x = np.eye(3)     # Identity matrix with n=m=3.
print(x)

If *m*, the number of columns, is not specified (optional), it defaults to *n*. Other arguments like the index of the diagonal or the array data type can also be specified.

- **`np.diag`** either builds a diagonal 2D array with given values along the diagonal, or builds a 1D array only containing the diagonal values. It accepts two arguments at most:
    - `v`: an array-like argument. If it's a 2D one, it returns a copy of its `k`-th diagonal; if it's a 1D one, it returns a 2D array with `v` on the `k`-th diagonal.
    - `k=0`: the diagonal index. 0: main; k>0, upper diagonals; k<0, lower diagonals.

In [ ]:
x = np.diag([1, 5, 4], k=1)  # Produces a 2D array with [1 5 4] in the first upper diagonal.
print(x)

x = np.diag([[1, 5, 6], [2, 3, 7]])  # Produces a 1D array with the elements from the main diagonal.
print(x)

##### **General ndarray Creation Functions**

They can be used to create arrays of any desired shape or dimensions. The most relevant ones are `zeros`, `ones`, `empty` and `random`. For example:

- **`np.zeros`** creates an array filled with **0s** with an specified `shape`.
- **`np.ones`** creates an array filled with **1s** with an specified `shape`. Identical to `zeros` in all aspects.
- **`empty`** will define an array of a specified shape and `dtype` of **uninitialized** values. Different from the previous ones, it does not initialize the values; the values stored will be arbitrary.

In [ ]:
x = np.zeros(5, dtype=np.float16)
print(x)

x = np.ones((3, 1))
print(x)

x = np.empty((3, 3), dtype=np.float32)
print(x)

There are many other routines to create arrays in a similar way as this: `random`, `full`, `zeros_like`, `ones_like`, `indices`...

#### **Editing existing arrays**

With arrays one has already created, new ones can be defined by replicating, joining or mutating the originals. For that, `np.copy` should be used, since the new array variable would be a **view** a not a **copy** of the original array's elements.

Other valid array creation methods include:

- **Reading from Files:** see [NumPy: Reading and Writing Files](https://numpy.org/doc/stable/user/how-to-io.html#how-to-io)
- **Using other libraries:** pandas, SciPy, etc.

### **NumPy Data-Types**

As seen before, the data-type attribute of an array describes how values are **represented** and **stored** in memory. 

There are many available numerical types in NumPy, more than in Python. They are actually **instances** of the `numpy.dtype` classes, and can be used to create arrays of any specified **dtype,** using the **scalar** types available in NumPy. 

The **scalar** types are arguments to the `dtype` keyword that many functions of this library accept. For example:

In [ ]:
x = np.arange(1, 5, 2, dtype=np.uint16)
x

While data types describe the **type** of data stored in an array, **scalar types** describe what kind of NumPy **object** is associated to the `dtype`, or each element is:

In [ ]:
x = np.array([10, 20, 30], dtype=np.int32)

# Array data-type.
print(x.dtype)

# Scalar type of an individual element.
print(type(x[0]))

# Scalar type associated with the dtype. 
print(x.dtype.type)  # Class representing any value of the array.


For type **conversion,** the method **`.astype()`** is used:

In [ ]:
x = x.astype(np.float32)
x

Conversion can also **overflow.** For example, when converting a 32-bit integer dtype to a 8-bit one, leading to unexpected values.

There are 5 basic numerical types representing each type of numerical data: Booleans (`bool`), integers (`int`), unsigned integers (`uint`), floating point numbers (`float`) and complex numbers (`complex`). There is a big amount of **built-in** scalar types, and even some of them are equivalent to normal Python types (see [NumPy: Built-in scalar types](https://numpy.org/doc/stable/reference/arrays.scalars.html#built-in-scalar-types)).

A basic numerical type name combined with a numeric **bitsize** defines a concrete type. The bitsize is the number of bits that are needed to represent a single value in **memory.**

The most common ones are:

| Numerical Type | Description/Value |
|---|---|
| **`int8`** | Alias of `byte`. 8-bit signed integer (`-128` to `127`). |
| **`int16`** | Alias of `short`. 16-bit signed integer (`-32_768` to `32_767`). |
| **`int32`** | Alias of `intc`. 32-bit signed integer (`-2`<sup>`31`</sup> to `2`<sup>`31`</sup>-1). |
| **`int64`** | Alias of `int_`. 64-bit signed integer (`-2`<sup>`63`</sup> to `2`<sup>63`</sup>-1). |
|     |     |
| **`uint8`** | Alias of `ubyte`. 8-bit **un**signed integer (`0` to `255`). |
| **`uint16`** | Alias of `ushort`. 16-bit **un**signed integer (`0` to `65_535`). |
| **`uint32`** | Alias of `uintc`. 32-bit **un**signed integer (`0` to `2`<sup>`32`</sup>-1). |
| **`uint64`** | Alias of `uint`. 64-bit **un**signed integer (`0` to `2`<sup>`64`</sup>-1). |
|     |     |
| **`float16`** | Alias of `half`. Half(16-bit)-precision floating-point number. See [**float16**](https://en.wikipedia.org/wiki/Half-precision_floating-point_format) for more info. |
| **`float32`** | Alias of `single`. Single(32-bit)-precision floating-point number. See [**float32**](https://en.wikipedia.org/wiki/Single-precision_floating-point_format) for more info. | 
| **`float64`** | Alias of `double`. Double(64-bit)-precision floating-point number. See [**float64**](https://en.wikipedia.org/wiki/Double-precision_floating-point_format) for more info. |

There are other types such as `bool` (alias of `bool`; Boolean value: True or False); or `complex64` (or `csingle`; complex number made of two float32's), among others. For every built-in scalar, see [NumPy Scalar types](https://numpy.org/doc/stable/reference/arrays.scalars.html#scalars).

#### **Type Promotion**

If one ends up mixing different data-types, NumPy will try to determine the most appropiate data-type for the resulting data. This is known as **promotion.** Most times, no action will be needed since the promotion will apply taking into account the precision of the input.

Of course, when we have inputs of the same `dtype`, the resulting one will match it, but, mixing two different `dtypes` will try to produce a result with the dtype of the input with **higher** precision:

In [ ]:
np.float16(3.4) + np.float16(5.8) # Same 'dtype' inputs. Result should match it.

In [ ]:
np.int16(11) + np.int32(45) # Different 'dtype' inputs. Highest precision: 32-bit integer.

There are clear **rules** that guide the promotion process. As mentioned above, numerical data-types can be grouped into four "kinds": `uint`, `int`, `float` and `complex`;  plus they have an associated precision (the **bitsize:** 8, 16, 32...).

In the end, the result of promotion will always be of an equal or higher kind of any of the inputs and have a precision greater than or equal to those of the inputs. For the complete guide, see [NumPy Type Promotion.](https://numpy.org/doc/stable/reference/arrays.promotion.html#numerical-promotion)

### **References**

- NumPy Docs: [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html), [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html), [Array Creation](https://numpy.org/doc/stable/user/basics.creation.html), [Data Types](https://numpy.org/doc/stable/user/basics.types.html).
- Python Data Science Handbook, Chapter 2.1: [Understanding Data Types in Python](https://jakevdp.github.io/PythonDataScienceHandbook/02.01-understanding-data-types.html).